# GROUP25 CENG313 — Baseline submission (LightGBM + char TF-IDF blend)
Inputs: the competition data **and** our dataset with the `GROUP25_CENG313` code. Internet: **off**.
Trains on `train.parquet` (~5-10 min on CPU) and writes `submission.csv` for the (hidden) test set.


In [ ]:
import sys, glob, shutil
from pathlib import Path

# 1) Find our package among the attached Kaggle datasets.
#    Works whether the dataset contains the folder GROUP25_CENG313/ or just its .py files.
pkg = None
for init in glob.glob("/kaggle/input/**/__init__.py", recursive=True):
    folder = Path(init).parent
    if (folder / "models.py").exists() and (folder / "features.py").exists():
        pkg = folder
        break
if pkg is None and Path("GROUP25_CENG313").exists():      # running locally from repo root
    pkg = Path("GROUP25_CENG313")
assert pkg is not None, "Package not found: attach the dataset that contains our .py files"
target = Path("/kaggle/working/GROUP25_CENG313") if Path("/kaggle/working").exists() else None
if target is not None:
    shutil.copytree(pkg, target, dirs_exist_ok=True)
    sys.path.insert(0, "/kaggle/working")
else:
    sys.path.insert(0, str(pkg.resolve().parent))
print("package:", pkg)

# 2) Find the competition data
hits = glob.glob("/kaggle/input/**/train.parquet", recursive=True)
DATA_DIR = Path(hits[0]).parent if hits else None
print("data:", DATA_DIR)

from GROUP25_CENG313 import data as D
from GROUP25_CENG313.models import FeatureModelBaseline, TfidfDiffBaseline


In [ ]:
train, test = D.load_competition_data(DATA_DIR)
y = D.encode_target(train["winner"])
print(train.shape, test.shape)

lgbm = FeatureModelBaseline("lgbm").fit(train, y)
tfidf = TfidfDiffBaseline().fit(train, y)
p = 0.5 * lgbm.predict_proba(test) + 0.5 * tfidf.predict_proba(test)
sub = D.make_submission(test["id"], p, "submission.csv")
print(sub["winner"].value_counts())
sub.head()